# MaternIA Integral - Entrenamiento en Colab

Entrena el modelo materno y el modelo fetal, y descarga los artefactos para copiarlos al proyecto.

In [ ]:
!pip install -q pandas numpy scikit-learn imbalanced-learn lightgbm ucimlrepo matplotlib joblib shap

In [ ]:
import os, json, joblib, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
warnings.filterwarnings("ignore")
RANDOM_STATE=42
os.makedirs("models", exist_ok=True); os.makedirs("reports", exist_ok=True); os.makedirs("docs", exist_ok=True)

In [ ]:
MATERNAL_FEATURES=["Age","SystolicBP","DiastolicBP","BS","BodyTemp","HeartRate"]
FETAL_FEATURES=["baseline_value","accelerations","fetal_movement","uterine_contractions","light_decelerations","severe_decelerations","prolongued_decelerations","abnormal_short_term_variability","mean_short_term_variability","percentage_of_time_with_abnormal_long_term_variability","mean_value_of_long_term_variability","histogram_width","histogram_min","histogram_max","histogram_number_of_peaks","histogram_number_of_zeroes","histogram_mode","histogram_mean","histogram_median","histogram_variance","histogram_tendency"]
FETAL_CLASS_NAMES={1:"Normal",2:"Sospechoso",3:"Patológico"}
MATERNAL_CLASS_NAMES={"low risk":"Bajo","mid risk":"Medio","high risk":"Alto"}
FETAL_ALIASES={"LB":"baseline_value","AC":"accelerations","FM":"fetal_movement","UC":"uterine_contractions","DL":"light_decelerations","DS":"severe_decelerations","DP":"prolongued_decelerations","ASTV":"abnormal_short_term_variability","MSTV":"mean_short_term_variability","ALTV":"percentage_of_time_with_abnormal_long_term_variability","MLTV":"mean_value_of_long_term_variability","Width":"histogram_width","Min":"histogram_min","Max":"histogram_max","Nmax":"histogram_number_of_peaks","Nzeros":"histogram_number_of_zeroes","Mode":"histogram_mode","Mean":"histogram_mean","Median":"histogram_median","Variance":"histogram_variance","Tendency":"histogram_tendency"}
def norm_risk(v):
    t=str(v).strip().lower().replace('_',' ')
    if t in ['low','low risk','bajo','riesgo bajo']: return 'low risk'
    if t in ['mid','medium','mid risk','medium risk','medio','riesgo medio']: return 'mid risk'
    if t in ['high','high risk','alto','riesgo alto']: return 'high risk'
    return t
def standardize_fetal(df):
    return df.rename(columns={c:FETAL_ALIASES.get(str(c).strip(), str(c).strip()) for c in df.columns}).copy()

In [ ]:
def eval_model(name, model, Xtr, ytr, Xte, yte, critical):
    ptr=model.predict(Xtr); pte=model.predict(Xte)
    return {"Modelo":name,"Accuracy Test":round(accuracy_score(yte,pte),4),"Balanced Accuracy Test":round(balanced_accuracy_score(yte,pte),4),"Precision Macro Test":round(precision_score(yte,pte,average='macro',zero_division=0),4),"Recall Macro Test":round(recall_score(yte,pte,average='macro',zero_division=0),4),"F1 Macro Test":round(f1_score(yte,pte,average='macro',zero_division=0),4),"Recall Clase Crítica Test":round(recall_score(yte,pte,labels=[critical],average='macro',zero_division=0),4)}, pte
def build_models():
    return {
      "LogisticRegression": Pipeline([("smote",SMOTE(random_state=RANDOM_STATE)),("scaler",StandardScaler()),("clf",LogisticRegression(max_iter=3000,class_weight='balanced',random_state=RANDOM_STATE))]),
      "RandomForest": Pipeline([("smote",SMOTE(random_state=RANDOM_STATE)),("clf",RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=RANDOM_STATE))]),
      "GradientBoosting": Pipeline([("smote",SMOTE(random_state=RANDOM_STATE)),("clf",GradientBoostingClassifier(random_state=RANDOM_STATE))]),
      "LightGBM": Pipeline([("smote",SMOTE(random_state=RANDOM_STATE)),("clf",LGBMClassifier(n_estimators=300,learning_rate=0.05,num_leaves=31,class_weight='balanced',random_state=RANDOM_STATE,verbose=-1))])
    }

In [ ]:
def train_task(X, y, task, critical, labels, display_labels, class_names, out_model):
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=RANDOM_STATE,stratify=y)
    rows=[]; fitted={}; preds={}
    for name, model in build_models().items():
        print('Entrenando', task, name)
        model.fit(Xtr,ytr)
        m,p=eval_model(name,model,Xtr,ytr,Xte,yte,critical)
        print(m)
        rows.append(m); fitted[name]=model; preds[name]=p
    df=pd.DataFrame(rows).sort_values(['Recall Clase Crítica Test','F1 Macro Test','Balanced Accuracy Test'],ascending=False).reset_index(drop=True)
    best=df.loc[0,'Modelo']; model=fitted[best]; pred=preds[best]
    prefix='maternal' if 'Maternal' in task else 'fetal'
    df.to_csv(f'reports/{prefix}_metrics_report_ordered.csv', index=False)
    with open(f'reports/{prefix}_classification_report_final.txt','w',encoding='utf-8') as f:
        f.write(f'Modelo seleccionado: {best}\n\n')
        f.write(classification_report(yte,pred,labels=labels,target_names=display_labels,zero_division=0))
    cm=confusion_matrix(yte,pred,labels=labels); disp=ConfusionMatrixDisplay(cm,display_labels=display_labels); disp.plot(values_format='d'); plt.title(task); plt.tight_layout(); plt.savefig(f'reports/{prefix}_confusion_matrix.png',dpi=300); plt.show()
    clf=model.named_steps['clf']
    if hasattr(clf,'feature_importances_'): scores=clf.feature_importances_
    elif hasattr(clf,'coef_'): scores=np.mean(np.abs(clf.coef_),axis=0)
    else: scores=np.zeros(X.shape[1])
    pd.DataFrame({'variable':X.columns,'importancia':scores}).sort_values('importancia',ascending=False).to_csv(f'reports/{prefix}_feature_importance.csv',index=False)
    artifact={'project':'MaternIA Integral','task':task,'model_name':best,'model':model,'feature_names':list(X.columns),'class_names':class_names,'labels':labels,'metrics':df.to_dict(orient='records')}
    joblib.dump(artifact, out_model)
    return df

In [ ]:
# Modelo materno
mat=fetch_ucirepo(id=863)
Xm=mat.data.features[MATERNAL_FEATURES].apply(pd.to_numeric, errors='coerce')
ym=mat.data.targets.iloc[:,0].apply(norm_risk)
print(Xm.shape, ym.value_counts())
df_m=train_task(Xm, ym, 'Maternal Health Risk', 'high risk', ['low risk','mid risk','high risk'], ['Bajo','Medio','Alto'], MATERNAL_CLASS_NAMES, 'models/maternIA_maternal_risk_model.pkl')
df_m

In [ ]:
# Modelo fetal
ctg=fetch_ucirepo(id=193)
df=standardize_fetal(pd.concat([ctg.data.features, ctg.data.targets], axis=1))
Xf=df[FETAL_FEATURES].apply(pd.to_numeric, errors='coerce')
yf=pd.to_numeric(df['NSP'], errors='coerce').astype(int)
print(Xf.shape, yf.value_counts())
df_f=train_task(Xf, yf, 'Fetal Health / Cardiotocography', 3, [1,2,3], ['Normal','Sospechoso','Patológico'], FETAL_CLASS_NAMES, 'models/maternIA_fetal_health_model.pkl')
df_f

In [ ]:
# Documentación del segundo modelo y compresión
Path('docs/resumen_integral.md').write_text('MaternIA Integral usa dos modelos funcionales: riesgo materno y estado fetal. No fusiona datasets; integra salidas mediante triaje.', encoding='utf-8')
!zip -r MaternIA_resultados_integral.zip models reports docs > /dev/null
from google.colab import files
files.download('MaternIA_resultados_integral.zip')